# Vanilla GAN - Synthetic Food Image Generation

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt

manualSeed = 42
torch.manual_seed(manualSeed)

## 1. Configuration and Data Loading

In [6]:
# Path to the dataset's image directory
image_dir = os.path.join("dataset", "food-101", "food-101", "images")

batch_size = 128
image_size = 64
nc = 3          # Channels (RGB)
nz = 100        # Latent vector size
num_epochs = 5  
lr = 0.0002
beta1 = 0.5     

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = torchvision.datasets.ImageFolder(root=image_dir, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

Using device: cpu


## 2. Model Architecture

In [7]:
class VanillaGenerator(nn.Module):
    def __init__(self):
        super(VanillaGenerator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(nz, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, nc * image_size * image_size),
            nn.Tanh()
        )

    def forward(self, input):
        input = input.view(input.size(0), -1)
        output = self.main(input)
        return output.view(-1, nc, image_size, image_size)

class VanillaDiscriminator(nn.Module):
    def __init__(self):
        super(VanillaDiscriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(nc * image_size * image_size, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, input):
        input = input.view(input.size(0), -1)
        return self.main(input)

## 3. Training Function

In [8]:
def train_gan(generator, discriminator, dataloader):
    criterion = nn.BCELoss()
    fixed_noise = torch.randn(64, nz, device=device)
    
    real_label = 1.
    fake_label = 0.
    
    optimizerD = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizerG = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
    
    img_list = []
    G_losses = []
    D_losses = []
    
    print("Starting Training...")
    for epoch in range(num_epochs):
        for i, data in enumerate(dataloader, 0):
            # Update D
            discriminator.zero_grad()
            real_cpu = data[0].to(device)
            b_size = real_cpu.size(0)
            label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
            output = discriminator(real_cpu).view(-1)
            errD_real = criterion(output, label)
            errD_real.backward()

            noise = torch.randn(b_size, nz, device=device)
            fake = generator(noise)
            label.fill_(fake_label)
            output = discriminator(fake.detach()).view(-1)
            errD_fake = criterion(output, label)
            errD_fake.backward()
            errD = errD_real + errD_fake
            optimizerD.step()

            # Update G
            generator.zero_grad()
            label.fill_(real_label)  
            output = discriminator(fake).view(-1)
            errG = criterion(output, label)
            errG.backward()
            optimizerG.step()
            
            if i % 100 == 0:
                print(f"[{epoch}/{num_epochs}][{i}/{len(dataloader)}] Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f}")
                
            G_losses.append(errG.item())
            D_losses.append(errD.item())
            
        with torch.no_grad():
            fake = generator(fixed_noise).detach().cpu()
        img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
        
    return G_losses, D_losses, img_list

## 4. Run Training

In [9]:
netG = VanillaGenerator().to(device)
netD = VanillaDiscriminator().to(device)
vanilla_G_losses, vanilla_D_losses, vanilla_img_list = train_gan(netG, netD, dataloader)

Starting Training...
[0/5][0/790] Loss_D: 1.3889 Loss_G: 0.6685
[0/5][100/790] Loss_D: 0.3525 Loss_G: 1.7868
[0/5][200/790] Loss_D: 0.2767 Loss_G: 3.1565
[0/5][300/790] Loss_D: 0.0451 Loss_G: 3.7575


KeyboardInterrupt: 